<a href="https://colab.research.google.com/github/AmarnathaGowda/50-Days-ML-Interview-Prep/blob/main/phi_2_quantization_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run this cell first to install dependencies

In [ ]:
!pip install torch transformers numpy sentencepiece accelerate

In [ ]:
!pip install bitsandbytes

In [ ]:
# Cell 2: Import Dependencies and Setup
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
from typing import List, Dict
import logging
import os

In [ ]:
# Import bitsandbytes for 8-bit loading (updated)
import bitsandbytes as bnb


In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [ ]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Using device: {device}")

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.5.1+cu124
CUDA available: True
CUDA device: Tesla T4


In [ ]:
# Cell 3: Model Configuration
class ModelConfig:
    def __init__(
        self,
        model_name: str = "microsoft/phi-2",
        max_length: int = 512,
        temperature: float = 0.7,
        top_p: float = 0.9,
        device: str = device
    ):
        """
        Configuration for the message response generator.

        Args:
            model_name (str): Name of the model to use
            max_length (int): Maximum length for input sequences
            temperature (float): Sampling temperature (higher = more random)
            top_p (float): Nucleus sampling parameter
            device (str): Device to run the model on
        """
        self.model_name = model_name
        self.max_length = max_length
        self.temperature = temperature
        self.top_p = top_p
        self.device = device

        # Define tone prompts
        self.tone_prompts = {
            "formal": "Respond in a formal and professional tone: ",
            "casual": "Respond in a casual and friendly tone: ",
            "polite": "Respond in a polite and respectful tone: ",
            "humorous": "Respond in a humorous and light-hearted tone: "
        }

In [ ]:
# Create default configuration
config = ModelConfig()
logger.info(f"Created configuration with model: {config.model_name}")

In [ ]:
# Import the correct class from bitsandbytes
# from transformers import BitsAndBytesConfig
#from bitsandbytes.cuda_setup.main import BitsAndBytesConfig as bnb
import bitsandbytes as bnb
from transformers import BitsAndBytesConfig

In [ ]:
# Cell 4: Model Loading Class
class ModelLoader:
    def __init__(self, config: ModelConfig):
        """
        Initialize the model loader with configuration.

        Args:
            config (ModelConfig): Configuration object
        """
        self.config = config
        self.model = None
        self.tokenizer = None

    def load_model(self) -> None:
        """
        Load the model and tokenizer from HuggingFace.
        Uses half-precision (float16) for efficiency.
        """
        try:
            logger.info(f"Loading model: {self.config.model_name}")

            # Load tokenizer
            self.tokenizer = AutoTokenizer.from_pretrained(
                self.config.model_name,
                trust_remote_code=True
            )
            logger.info("Tokenizer loaded successfully")

            # # Load model with optimizations (float 16 bit)
            # self.model = AutoModelForCausalLM.from_pretrained(
            #     self.config.model_name,
            #     torch_dtype=torch.float16,
            #     device_map=self.config.device,
            #     trust_remote_code=True
            # )



              # Load model with 8-bit quantization using bitsandbytes
            self.model = AutoModelForCausalLM.from_pretrained(
                self.config.model_name,
                # load_in_8bit=True,  # Enable 8-bit loading
                device_map=self.config.device,
                trust_remote_code=True,
                quantization_config=BitsAndBytesConfig(
                    load_in_8bit=True,
                    llm_int8_threshold=6.0  # Adjust threshold if needed
                )
            )
            logger.info("Model loaded successfully")



            # Move model to device
            # self.model.to(self.config.device)

            # Print model size information
            model_size = sum(p.numel() for p in self.model.parameters()) * 2 / (1024 * 1024)
            logger.info(f"Model size (FP16): {model_size:.2f} MB")

        except Exception as e:
            logger.error(f"Error loading model: {str(e)}")
            raise

    def verify_model(self) -> bool:
        """
        Verify model loading with a simple test generation.

        Returns:
            bool: True if verification successful
        """
        try:
            test_input = "Hello, how are you?"
            inputs = self.tokenizer(test_input, return_tensors="pt").to(self.config.device)

            with torch.no_grad():
                outputs = self.model.generate(
                    inputs["input_ids"],
                    max_length=50,
                    num_return_sequences=1
                )

            decoded = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            logger.info("Model verification successful")
            return True

        except Exception as e:
            logger.error(f"Model verification failed: {str(e)}")
            return False

In [ ]:
# Cell 5: Test Model Loading
if __name__ == "__main__":
    # Create model loader
    loader = ModelLoader(config)

    # Load and verify model
    loader.load_model()
    verification_result = loader.verify_model()

    print(f"\nModel setup {'successful' if verification_result else 'failed'}")
    print(f"Model ready on device: {config.device}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Model setup successful
Model ready on device: cuda


In [ ]:
#Import Additional Dependencies
import torch.quantization
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
from typing import Dict, List, Optional, Tuple

In [ ]:
 #Cell 2: Model Quantization Class
class ModelQuantizer:
    def __init__(self, model: AutoModelForCausalLM):
        """
        Initialize the quantizer with a loaded model.

        Args:
            model: Loaded PyTorch model
        """
        self.model = model
        self.quantized_model = None

    def quantize_dynamic(self) -> None:
        """
        Perform dynamic quantization on the model.
        Converts weights to int8 while keeping activations in float16.
        """
        try:
            logger.info("Starting dynamic quantization")

            # # Configure quantization
            # self.model.qconfig = torch.quantization.get_default_dynamic_qconfig()

            # Configure quantization using torch.ao.quantization.qconfig
            self.model.qconfig = torch.ao.quantization.qconfig.default_dynamic_qconfig

            # Prepare for quantization
            torch.quantization.prepare(self.model, inplace=True)

            # Quantize the model
            self.quantized_model = torch.quantization.quantize_dynamic(
                self.model,
                {torch.nn.Linear},  # Quantize only linear layers
                dtype=torch.qint8
            )

            # Calculate and log model sizes
            original_size = sum(p.numel() for p in self.model.parameters()) * 2  # FP16
            quantized_size = sum(p.numel() for p in self.quantized_model.parameters())

            logger.info(f"Original model size (FP16): {original_size / 1024 / 1024:.2f} MB")
            logger.info(f"Quantized model size (INT8): {quantized_size / 1024 / 1024:.2f} MB")
            logger.info(f"Size reduction: {(1 - quantized_size/original_size) * 100:.1f}%")

        except Exception as e:
            logger.error(f"Quantization failed: {str(e)}")
            raise

    def benchmark_inference(self, tokenizer: AutoTokenizer, input_text: str, num_runs: int = 5) -> Tuple[float, float]:
        """
        Benchmark inference speed for original and quantized models.

        Args:
            tokenizer: Model tokenizer
            input_text: Sample input for testing
            num_runs: Number of inference runs for averaging

        Returns:
            Tuple of (original_time, quantized_time) in seconds
        """
        inputs = tokenizer(input_text, return_tensors="pt").to(self.model.device)

        # Benchmark original model
        original_times = []
        for _ in range(num_runs):
            start_time = time.time()
            with torch.no_grad():
                self.model.generate(
                    inputs["input_ids"],
                    max_length=50,
                    num_return_sequences=1
                )
            original_times.append(time.time() - start_time)

        # Benchmark quantized model
        quantized_times = []
        for _ in range(num_runs):
            start_time = time.time()
            with torch.no_grad():
                self.quantized_model.generate(
                    inputs["input_ids"],
                    max_length=50,
                    num_return_sequences=1
                )
            quantized_times.append(time.time() - start_time)

        avg_original = sum(original_times) / len(original_times)
        avg_quantized = sum(quantized_times) / len(quantized_times)

        logger.info(f"Average inference time (original): {avg_original:.3f}s")
        logger.info(f"Average inference time (quantized): {avg_quantized:.3f}s")
        logger.info(f"Speed improvement: {((avg_original/avg_quantized) - 1) * 100:.1f}%")

        return avg_original, avg_quantized

In [ ]:
# Cell 3: Response Generator Class
class ResponseGenerator:
    def __init__(self, model: AutoModelForCausalLM, tokenizer: AutoTokenizer, config: ModelConfig):
        """
        Initialize the response generator.

        Args:
            model: Loaded (and optionally quantized) model
            tokenizer: Model tokenizer
            config: Model configuration
        """
        self.model = model
        self.tokenizer = tokenizer
        self.config = config

    def generate_response(
        self,
        message: str,
        tone: str = "casual",
        max_length: Optional[int] = None
    ) -> str:
        """
        Generate a response for a given message and tone.

        Args:
            message: Input message
            tone: Desired tone of response
            max_length: Maximum length of generated response

        Returns:
            Generated response string
        """
        try:
            # Get tone prompt and create full prompt
            tone_prompt = self.config.tone_prompts.get(tone, self.config.tone_prompts["casual"])
            full_prompt = f"Message: {message}\n{tone_prompt}"

            # Tokenize input
            inputs = self.tokenizer(
                full_prompt,
                return_tensors="pt",
                truncation=True,
                max_length=self.config.max_length
            ).to(self.config.device)

            # Generate response
            with torch.no_grad():
                outputs = self.model.generate(
                    inputs["input_ids"],
                    max_length=max_length or self.config.max_length,
                    num_return_sequences=1,
                    temperature=self.config.temperature,
                    top_p=self.config.top_p,
                    pad_token_id=self.tokenizer.eos_token_id,
                    do_sample=True
                )

            # Decode and clean response
            response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            response = response.replace(full_prompt, "").strip()

            return response

        except Exception as e:
            logger.error(f"Error generating response: {str(e)}")
            raise

    def generate_multi_tone_responses(
        self,
        message: str,
        tones: Optional[List[str]] = None
    ) -> Dict[str, str]:
        """
        Generate responses in multiple tones.

        Args:
            message: Input message
            tones: List of tones to use (defaults to all available tones)

        Returns:
            Dictionary mapping tones to responses
        """
        if tones is None:
            tones = list(self.config.tone_prompts.keys())

        responses = {}
        for tone in tones:
            try:
                response = self.generate_response(message, tone)
                responses[tone] = response
            except Exception as e:
                logger.error(f"Error generating {tone} response: {str(e)}")
                responses[tone] = f"Error generating response: {str(e)}"

        return responses


In [ ]:
# Cell 4: Test Quantization and Response Generation
def test_pipeline():
    # Load model first (assuming Part 1 is already run)
    config = ModelConfig()
    loader = ModelLoader(config)
    loader.load_model()

    # Quantize model
    quantizer = ModelQuantizer(loader.model)
    quantizer.quantize_dynamic()

    # Benchmark inference
    test_message = "Can we schedule a meeting for tomorrow?"
    original_time, quantized_time = quantizer.benchmark_inference(
        loader.tokenizer,
        test_message
    )

    # Create response generator with quantized model
    generator = ResponseGenerator(
        quantizer.quantized_model,
        loader.tokenizer,
        config
    )

    # Test response generation
    test_responses = generator.generate_multi_tone_responses(test_message)

    # Print results
    print("\nTest Responses:")
    for tone, response in test_responses.items():
        print(f"\n{tone.upper()} TONE:")
        print(response)



In [ ]:
if __name__ == "__main__":
    test_pipeline()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

ERROR:__main__:Quantization failed: Embedding quantization is only supported with float_qparams_weight_only_qconfig.


AssertionError: Embedding quantization is only supported with float_qparams_weight_only_qconfig.